#  IMPORTS



In [1]:

# use the dpivsoft conda environment
# Standar libraries
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import dpivsoft.Postprocessing as post #necessary for the vorticity calculcation comment if you don't want to install it
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.io import loadmat
import pandas as pd


%matplotlib qt



# Functions data

In [2]:
def load_piv(file_path: str):
    mat = loadmat(file_path)
      

    if 'wienerwurst' in mat.keys(): #file_path.endswith('batchprocess.mat'):
        
        # Coordinates
        x_all = mat['x']
        y_all = mat['y']
        X_original = x_all[:,:,0]
        Y_original = y_all[:,:,0]

        # Cell arrays: pick the frame cell
        U_filtered = mat['u_filt'][:,:,:]
        V_filtered = mat['v_filt'][:,:,:]
        U_org = mat['u'][:,:,:] #velocity prior to validation
        V_org = mat['v'][:,:,:] #velocity prior to validation

        # Flip axes
        X = X_original[:, ::-1]
        Y = Y_original[::-1, :]
        U = U_filtered[::-1, ::-1,:] #velocity after validation
        V = V_filtered[::-1, ::-1,:] #velocity after validation
        U_original = U_org[::-1, ::-1,:] #velocity prior to validation
        V_original = V_org[::-1, ::-1,:] #velocity prior to validation

        nframes=mat['u_filt'].shape[2]
    
    elif 'calxy' in mat.keys():
        
        # Coordinates
        x_all = mat['x']
        y_all = mat['y']
        X_original = x_all[0, 0]
        Y_original = y_all[0, 0]

        nframes=mat['u_filtered'].shape[0]

        tmp=np.zeros((X_original.shape[0],X_original.shape[1],nframes))
        U_filtered = tmp.copy()
        V_filtered = tmp.copy()
        U_org = tmp.copy()
        V_org = tmp.copy()

        for i in range(nframes):
            U_filtered[:,:,i] = mat['u_filtered'][i,0]
            V_filtered[:,:,i] = mat['v_filtered'][i,0]
            U_org[:,:,i] = mat['u_original'][i,0]
            V_org[:,:,i] = mat['v_original'][i,0]
        
        # Flip axes
        X = X_original[:, ::-1]
        Y = Y_original[::-1, :]
        U = U_filtered[::-1, ::-1,:] #velocity after validation
        V = V_filtered[::-1, ::-1,:] #velocity after validation
        U_original = U_org[::-1, ::-1,:] #velocity prior to validation
        V_original = V_org[::-1, ::-1,:] #velocity prior to validation
    
    else: print('Unknown PIV file')

    # Mask wherever original vectors are NaN
    mask = np.isnan(U_original) | np.isnan(V_original)
    # Ensure float dtype to hold NaNs
    U = U.astype(float, copy=False)
    V = V.astype(float, copy=False)
    X = X.astype(float, copy=False)
    Y = Y.astype(float, copy=False)
    # Apply mask elementwise
    U[mask] = np.nan
    V[mask] = np.nan
    # X[mask] = np.nan
    # Y[mask] = np.nan
    Y=Y.max()-Y
    V=-V

    return X, Y, U, V, U_original, V_original, X_original, Y_original,nframes

def getMagnitude(x,y,u,v):
    """Compute velocity magnitude."""
    vel = np.sqrt(u**2 + v**2)
    return x,y,vel

def getVorticity(x,y,u,v):
    """Compute vorticity."""
    X, Y, w = post.vorticity(x, y, u, v, 'circulation')
    return X,Y,w

def getDivergence(x,y,u,v):
    """Compute divergence."""
    divergence = post.divergence(x,y,u,v)
    return x,y,divergence

def getTimeAveraged(dirResults='./',dt=1):
    files = os.listdir(dirResults)
    files = sorted([i for i in files if i.endswith('.npz')]);
    #print(files)

    for i in range(len(files)):
        x,y,u,v, t = load_piv(dirResults,i, res_files=files,dt=dt)
        if i==0:
            xvort,yvort,vorticity=getVorticity(x,y,u,v)
            vorticity_mean = np.zeros(vorticity.shape)
            magnitude_mean = np.zeros(u.shape)
            u_mean = np.zeros(u.shape)
            v_mean = np.zeros(v.shape)
        u_mean += u
        v_mean += v
        x,y,magnitude=getMagnitude(x,y,u,v)
        magnitude_mean+= magnitude
        xvort,yvort,vorticity=post.vorticity(x, y, u, v, 'circulation')
        vorticity_mean+= vorticity

    magnitude_mean/=len(files)  
    vorticity_mean/=len(files)
    u_mean/=len(files)
    v_mean/=len(files)
    timeAveraged=len(files)*dt

    return x,y,u_mean,v_mean,magnitude_mean,xvort,yvort,vorticity_mean,timeAveraged

def create_mask(x, y, pts_ROI):
    pts = np.asarray(pts_ROI).reshape(2, 2)
    x1, y1 = pts[0]
    x2, y2 = pts[1]

    xmin, xmax = sorted((x1, x2))
    ymin, ymax = sorted((y1, y2))

    mask = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
    return mask

def extractROI(x, y, u, v, mask):
    
    if not np.any(mask):
        return np.array([]), np.array([]), np.array([]), np.array([])

    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)

    x_roi = x[np.ix_(rows, cols)]
    y_roi = y[np.ix_(rows, cols)]
    u_roi=np.zeros((x_roi.shape[0], x_roi.shape[1], u.shape[2]))
    v_roi=np.zeros((x_roi.shape[0], x_roi.shape[1], v.shape[2]))

    for i in range(u.shape[2]):
        _u=np.copy(u[:,:,i])
        _v=np.copy(v[:,:,i])
        u_roi[:,:,i] = _u[np.ix_(rows, cols)]
        v_roi[:,:,i] = _v[np.ix_(rows, cols)]

    return x_roi, y_roi, u_roi, v_roi

def Win_BH(s,ratio):
    '''
    apply a BlacmanHarris window to the signal
    ratio=length of the 1/2 window/length of the signal. 
    1/2 apply to the beginning of the signal the other 1/2 
    at the end
    return s windowed
    '''

    import scipy.signal as sp


    L=len(s)
    LengthBH=int(L*ratio)
    Window=[1]*L
    winBH=np.zeros(2*LengthBH)    
    #winBH=sp.blackmanharris(2*LengthBH)
    winBH=sp.windows.blackman(2*LengthBH)
    for i in range(0,LengthBH):
        Window[i]=winBH[i]
        Window[L-1-i]=winBH[i]   
    sW=np.multiply(Window,s)
    return sW

def fftjn(s,t,fs,ratio, detrend=True, window=True ):
    '''
    fftjn(u,t,fs,ratio)
    calculate fft of s(t) sampled at fs 
    ratio is the ratio between the BlackmanHarris window length and signal length. 
    1/2 apply to the beginning of the signal the other 1/2 
    at the end detrend and windowing can be disabled using 
    fftjn(s,t,fs,ratio, detrend=False, window=False )

    returns The frequency and the FFT (complex number)
    '''
    
    import scipy.signal as sp
    import numpy as np


    L=len(t)
    if detrend:
        s=sp.detrend(s)
    if window:
        s=Win_BH(s,ratio)     
    sFFT = np.fft.fft(s)
    sFREQ = np.fft.fftfreq(len(t),1/fs)
    Nfreq = len(sFREQ)//2
    return np.abs(sFREQ[0:Nfreq]), np.abs(sFFT[0:Nfreq])

def densitySpectrum2D(x1,y1,z):
    '''
    densitySpectrum(x1,y1,z)
    Calculate the 2D density spectrum in space of z
    Input:
    X1,Y1, 1D array with coordinates
    z array (len(y1), len(x1))
    Output:
    kx,FFTx,ky,FFTy,k,FFT
    '''

    dx=np.mean(np.diff(x1))
    dy=-np.mean(np.diff(y1))
    _kx = np.fft.fftfreq(len(x1),dx)
    _ky = np.fft.fftfreq(len(y1),dy)
    _zfft=np.fft.fft2(z)
    _s=np.abs(_zfft)
    Nx = int(len(_kx)/2.)
    Ny = int(len(_ky)/2.)
    kx=_kx[:Nx]
    ky=_ky[:Ny]
    s=_s[0:Ny,0:Nx]
            
    ekx=np.mean(s,axis=0)
    eky=np.mean(s,axis=1)
    return kx,ekx,ky,eky,s


# Functions plots

In [3]:

def plotQuiver(x, y, u, v, img=None, isImg=False, scale=1, save=True,output_dir='./', time_stamp=0):
    """Plot quiver with arrows starting at grid points."""
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.quiver(x, y, u, v, scale=scale,zorder=1)
    ax.set_xlabel('x (m)',fontsize=18)
    ax.set_ylabel('y (m)',fontsize=18)
    if isImg:
        ax.imshow(img, cmap='gray', origin='upper', extent=[x.min(), x.max(), y.min(), y.max()], alpha=0.3,zorder=2)
    ax.set_title('Instantenous velocity Field at '+str(time_stamp)+ 's',fontsize=18)
    # Save the figure as PNG
    figname=output_dir+'Figure_velocityVector_at_'+str(time_stamp)+'s.png'
    if save==True:
        fig.savefig(figname, dpi=300, bbox_inches='tight')
    return fig, ax

def plotColormap(x, y, data, img=None, isImg=False,bounds=False, datamin=0, datamax=1, N=50, cmap='inferno'):
    """Plot colormap of velocity magnitude."""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    if bounds==True:
        c = ax.contourf(x,y,data, N, cmap=cmap, vmin=datamin, vmax=datamax,zorder =1)
        #c = ax.pcolormesh(x, y, vel, shading='auto', cmap='inferno', vmin=velmin, vmax=velmax,zorder=1)
    else:
        c = ax.contourf(x,y,data, N, cmap=cmap, zorder =1)
    #show the image in the background
    if isImg:
        ax.imshow(img, cmap='gray', origin='upper', extent=[x.min(), x.max(), y.min(), y.max()], alpha=0.5,zorder=2) 
    ax.set_xlabel('x (m)', fontsize=18)
    ax.set_ylabel('y (m)', fontsize=18)

    return fig, ax, c

def plotColormapVel(x, y, u, v, img=None, isImg=False, bounds=False, velmin=0, velmax=1, save=True,output_dir='./', time_stamp=0):
    """Plot colormap of velocity magnitude."""
    x,y,vel=getMagnitude(x,y,u,v)
    fig, ax = plt.subplots(figsize=(10, 6))
    
    #plot the velocity magnitude map
    if bounds==True:
        c = ax.contourf(x,y,vel, 50, cmap='inferno', vmin=velmin, vmax=velmax,zorder =1)
        #c = ax.pcolormesh(x, y, vel, shading='auto', cmap='inferno', vmin=velmin, vmax=velmax,zorder=1)
    else:
        c = ax.contourf(x,y,vel, 50, cmap='inferno', zorder =1)
        # c = ax.pcolormesh(x, y, vel, shading='gouraud', cmap='jet',zorder=1)
    #show the image in the background
    if isImg:
        ax.imshow(img, cmap='gray', origin='upper', extent=[x.min(), x.max(), y.min(), y.max()], alpha=0.5,zorder=2)
    
    ax.set_xlabel('x (m)', fontsize=18)
    ax.set_ylabel('y (m)', fontsize=18)
    ax.set_title('Instantenous velocity magnitude at '+str(time_stamp)+ 's',fontsize=18)
    # Make the colorbar the same height as the plot
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    fig.colorbar(c, cax=cax, label='Velocity Magnitude (u² + v²) in m/s at ' )
    if save==True:
        figname=output_dir+'Figure_velocityMagnitude_at_'+str(time_stamp)+'s.png'
        fig.savefig(figname, dpi=300, bbox_inches='tight')
    return fig, ax
    
def plotColormapVort(x, y,u,v,img=None, isImg=False,bounds=False, velmin=0, velmax=1, save=True,output_dir='./', time_stamp=0):
    """Plot colormap of velocity magnitude."""
    X, Y, w = getVorticity(x,y,u,v)
    fig, ax = plt.subplots(figsize=(10, 6))
    
    #plot the velocity magnitude map
    if bounds==True:
        c = ax.contourf(X, Y, w, 50, cmap='jet', vmin=velmin, vmax=velmax,zorder =1)
        #c = ax.pcolormesh(X, Y, w, shading='auto', cmap='inferno', vmin=velmin, vmax=velmax,zorder=1)
    else:
        c = ax.contourf(X, Y, w, 50, cmap='jet',zorder =1)
        # c = ax.pcolormesh(X, Y, w, shading='auto', cmap='inferno',zorder=1)
    #show the image in the background
    if isImg:
        ax.imshow(img, cmap='gray', origin='upper', extent=[x.min(), x.max(), y.min(), y.max()], alpha=0.5,zorder=2)
    
    
    ax.set_xlabel('x (m)', fontsize=18)
    ax.set_ylabel('y (m)', fontsize=18)
    ax.set_title('Instantenous vorticity at '+str(time_stamp)+ 's',fontsize=18)
    # Make the colorbar the same height as the plot
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    fig.colorbar(c, cax=cax, label='vorticity' )
    if save==True:
        figname=output_dir+'Figure_vorticity_at_'+str(time_stamp)+'s.png'
        fig.savefig(figname, dpi=300, bbox_inches='tight')
    return fig, ax



def plotColormapDiv(x, y,u,v,img=None, isImg=False,bounds=False, velmin=0, velmax=1, save=True,output_dir='./', time_stamp=0):
    """Plot colormap of velocity magnitude."""
    x,y,divergence = getDivergence(x,y,u,v)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    #plot the velocity magnitude map
    if bounds==True:
        c = ax.contourf(x,y,divergence, 50, cmap='jet', vmin=velmin, vmax=velmax,zorder =1)
        # c = ax.pcolormesh(x,y,divergence, shading='auto', cmap='jet', vmin=velmin, vmax=velmax,zorder=1)
    else:
        c = ax.contourf(x,y,divergence, 50, cmap='jet', zorder =1)
        # c = ax.pcolormesh(x,y,divergence, shading='auto', cmap='jet',zorder=1)
    #show the image in the background
    if isImg:
        ax.imshow(img, cmap='gray', origin='upper', extent=[x.min(), x.max(), y.min(), y.max()], alpha=0.5,zorder=2)


    ax.set_xlabel('x (m)', fontsize=18)
    ax.set_ylabel('y (m)', fontsize=18)
    ax.set_title('Instantenous divergence at '+str(time_stamp)+ 's',fontsize=18)
    # Make the colorbar the same height as the plot
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    fig.colorbar(c, cax=cax, label='divergence' )
    if save==True:
        figname=output_dir+'Figure_divergence_at_'+str(time_stamp)+'s.png'
        fig.savefig(figname, dpi=300, bbox_inches='tight')
    return fig, ax





# Directories and calibration parameters

In [4]:
# Define directories and file extensions
mainDir="/Users/jeromenoir/Documents/MyDocuments/LOCAL_PROJECT/TOPOGRAPHY_LIBRATION/CylinderExperimentsGMA/k20_bottomOnly/frot050_flib080_dphi2deg/"
dirPIV = mainDir
dirResults = mainDir + 'PostProcessing/'
dirImages  = mainDir
image_extension = 'tif'
PIV_file=dirPIV+'PIVlab_results_uncalibrated.mat'

# Setup parameters for calibration and ROI selection
doCalibration=True
xscale=1.2323e-4 #m/px
yscale=xscale# 9.5e-5 #m/px

fps=60
dt=1/fps#40e-3 #16.6667e-3 # s

# Selection of ROI
frame_indx=0 # frame to use for the ROI selection
isROISelection=False #Set to True if you want to select the ROI interactively by clicking on the plot
isROIextraction=True #Set to True if you want to extract the u,v,x,y values inside the rectangle defined by the pts_ROI either
                     #by clicking on the plot or by defining the coordinates of the rectangle in pts_ROI
                     #[(x0,y0),(x1,y1)] x0,y0 top left corner, x1,y1 bottom right corner
pts_ROI = [(0.02002585173778408, 0.13110555228124998),(0.1956196010465909, 0.0065950525471591)] #for calibrated data
# pts_ROI= [(250.24676136363615, 1078.9703125000003),(1736.0767045454547, 89.11681818181816)] #for uncalibrated data


############################### LOADING THE DATA #####################################################
#check the directory exists:

if not os.path.exists(dirPIV):
    print("Unknown Results directory" )

if not os.path.exists(PIV_file):
    print("PIV_file: %s File not found", PIV_file)

if not os.path.exists(dirImages):
    print("Unknown Imagedirectory" )
else:
    frame_files = os.listdir(dirImages)
    frame_files = sorted([i for i in frame_files if i.endswith(image_extension)])
    print("Total number of image pairs: ", int(len(frame_files)/2))
if not os.path.exists(dirResults):
    os.makedirs(dirResults, exist_ok=True)

#load the data
x, y, u, v, u_unfiltered, v_unfiltered,xorig,yorig,nframes = load_piv(PIV_file)
print("Data loaded successfully from ", PIV_file)
print("Number of frames: ", nframes)

#initialize the calibration flag
isCalibrated=False

if (doCalibration & isCalibrated==False):
    print("Data is uncalibrated, applying scale factors.")
    x=xscale*x
    y=yscale*y
    u=xscale*u/dt
    v=yscale*v/dt
    u_unfiltered=xscale*u_unfiltered/dt
    v_unfiltered=yscale*v_unfiltered/dt
    xorig=xscale*xorig
    yorig=yscale*yorig
    isCalibrated=True

print('resolution in x = ',np.abs(np.diff(x, axis=1).mean()))
print('resolution in y = ',np.abs(np.diff(y, axis=0).max()))

ImageFiles=dirImages+frame_files[1]
img = cv2.imread(ImageFiles, cv2.IMREAD_GRAYSCALE)

time_stamp=frame_indx*dt

if isROISelection:
    #Plot the velocity map and selesct two end points of the profile
    # figVel, axVel = plotQuiver(x, y, u, v, img, scale=80, save=False,output_dir=dirResults, time_stamp=time_stamp)
    figVel, axVel = plotColormapVel(x, y,u[:,:,0],v[:,:,0],img=img, isImg=True, bounds=False, velmin=0, velmax=1, save=True,output_dir=dirResults, time_stamp=time_stamp)
    axVel.set_title('Select two points to define the edge of the ROI',fontsize=18)
    pts_ROI = plt.ginput(2)
    plt.close(figVel)

if isROIextraction:
    mask = create_mask(x, y, pts_ROI)
    X, Y, U, V = extractROI(x, y, u, v, mask)
    figVel, axVel = plotColormapVel(X, Y, U[:,:,0], V[:,:,0],isImg=False, bounds=False, velmin=0, velmax=1, save=True, output_dir=dirResults, time_stamp=time_stamp)
    axVel.set_title('Velocity field in the selected ROI', fontsize=18)

else:
    X=x
    Y=y
    U=u
    V=v
    figVel, axVel = plotColormapVel(X, Y,U[:,:,0],V[:,:,0],img, isImg=False,bounds=False, velmin=0, velmax=1, save=True,output_dir=dirResults, time_stamp=time_stamp)
    axVel.set_title('Instantenous velocity magnitude at '+str(time_stamp)+ 's',fontsize=18)



Total number of image pairs:  500
Data loaded successfully from  /Users/jeromenoir/Documents/MyDocuments/LOCAL_PROJECT/TOPOGRAPHY_LIBRATION/CylinderExperimentsGMA/k20_bottomOnly/frot050_flib080_dphi2deg/PIVlab_results_uncalibrated.mat
Number of frames:  500
Data is uncalibrated, applying scale factors.
resolution in x =  0.00197168
resolution in y =  0.0019716800000000034


# Plot individual frames

## Plots


In [5]:
frame_indx=50


plotColormapVel(X, Y, U[:,:,frame_indx], V[:,:,frame_indx], bounds=False, velmin=0, velmax=1, save=False,output_dir='./', time_stamp=frame_indx*dt)
plotQuiver(X, Y, U[:,:,frame_indx], V[:,:,frame_indx], img=img, isImg=False, scale=0.2, save=True,output_dir='./', time_stamp=frame_indx*dt)
plotColormapVort(X, Y, U[:,:,frame_indx], V[:,:,frame_indx], bounds=False, velmin=0, velmax=1, save=False,output_dir='./', time_stamp=frame_indx*dt)


(<Figure size 2000x1200 with 2 Axes>,
 <Axes: title={'center': 'Instantenous vorticity at 0.8333333333333334s'}, xlabel='x (m)', ylabel='y (m)'>)

# Stacking the fields over a full period

In [6]:
flib=0.5
samplingPIV=fps/2

frame_interval = int(samplingPIV/flib)
U_stacked = np.zeros_like(U[:,:,0])
V_stacked = np.zeros_like(U[:,:,0])

for i in range(0, nframes, frame_interval):
    U_stacked += U[:,:,i]
    V_stacked += V[:,:,i]

U_stacked /= (nframes // frame_interval)
V_stacked /= (nframes // frame_interval)

plotColormapVel(X, Y, U_stacked, V_stacked, bounds=False, velmin=0, velmax=1, save=False,output_dir=mainDir, time_stamp=frame_indx*dt)
# plotColormapVel(X, Y, U[:,:,frame_indx], V[:,:,frame_indx], bounds=False, velmin=0, velmax=1, save=False,output_dir='./', time_stamp=frame_indx*dt)
# plotQuiver(X, Y, U_stacked, U_stacked, img=img, isImg=False, scale=0.2, save=True,output_dir='./', time_stamp=frame_indx*dt)
# plotQuiver(X, Y, U[:,:,frame_indx], V[:,:,frame_indx], img=img, isImg=False, scale=0.2, save=True,output_dir='./', time_stamp=frame_indx*dt)

(<Figure size 2000x1200 with 2 Axes>,
 <Axes: title={'center': 'Instantenous velocity magnitude at 0.8333333333333334s'}, xlabel='x (m)', ylabel='y (m)'>)

# Extract the mean kinetic energy within the ROI for a list of frames



In [7]:
frame_start=0
frame_end=nframes-1
output_file=dirResults+'KineticEnergy_timeSeries.npz'


Ek_frame=[]
t=[]
for indx_frame in range(frame_start,frame_end):
    time_stamp=indx_frame*dt
    Ek_frame.append(0.5*np.nanmean(U[:,:,indx_frame]**2 + V[:,:,indx_frame]**2))
    t.append(time_stamp)

Ek_frame=np.array(Ek_frame)
t=np.array(t)
plt.figure()
plt.plot(t,Ek_frame)
plt.xlabel('Time (s)')
plt.ylabel('Mean Kinetic Energy (m²/s²)')

print('Done processing frames.')
print('Mean kinetic energy %s = ',Ek_frame.mean())
print('standard deviation of kinetic energy %s = ',Ek_frame.std())


np.savez(output_file,
         dirResults=dirResults, 
         PIV_file=PIV_file, 
         dt=dt, 
         xscale=xscale, 
         yscale=yscale, 
         doCalibration=doCalibration, 
         pts_ROI=pts_ROI, 
         isROISelection=isROISelection, 
         isROIextraction=isROIextraction, 
         frame_start=frame_start, 
         frame_end=frame_end, 
         t=t, 
         Ek_frame=Ek_frame)

Done processing frames.
Mean kinetic energy %s =  1.1155258599495498e-05
standard deviation of kinetic energy %s =  1.1820856250331911e-06


# Calculate the FFT in space

For testing the FFT use the synthetic below

In [8]:
kx = 50.0
ky = 30.0
_U=np.cos(2*np.pi*kx*X)+np.cos(2*np.pi*ky*Y)
_V=np.cos(2*np.pi*kx*X)+np.cos(2*np.pi*ky*Y)
figVel, axVel = plotColormapVel(X, Y,_U,_V,img, isImg=False,bounds=False, velmin=0, velmax=1, save=False,output_dir=dirResults, time_stamp=time_stamp)

## horizontal FFT

### for a single frame

In [9]:
frame_indx=0
output_file=dirResults+'FFT_horizontal_singleFrame.npz'

_X=X[0, :]
_Y=Y[:, 0]
Ny=len(_Y)
Nx=len(_X)

#select the field for the FFT calculation
Z = U_stacked[:,:]

#Initialize an array to hold the FFT results
fft_along_x=[]
for i in range(Ny):
    _U1D=Z[i,:]    
    f,e = fftjn(_U1D, _X, 1/(_X[1]-_X[0]), 0.5, detrend=True, window=True)
    fft_along_x.append(e)
fft_along_x=np.array(fft_along_x)


fgrid,ygrid=np.meshgrid(f,_Y)
plt.figure()
plt.contourf(fgrid,ygrid,fft_along_x, 50, cmap='inferno', zorder =1)
plt.xlabel('Frequency (1/m)')
plt.ylabel('y (m)')
plt.title('FFT along x-axis')

fft_along_x_stacked = np.mean(fft_along_x, axis=0)
plt.figure()
plt.plot(f, fft_along_x_stacked)
plt.xlabel('Frequency (1/m)')
plt.ylabel('Average FFT Magnitude')
plt.title('Average FFT along x-axis')


Text(0.5, 1.0, 'Average FFT along x-axis')

### averaged in time

In [46]:
frame_start=0
frame_end=1#nframes-1
output_file=dirResults+'FFT_horizontal_timeAveraged.npz'

_X=X[0, :]
_Y=Y[:, 0]
Ny=len(_Y)
Nx=len(_X)

#select the field for the FFT calculation
Z = V

#Initialize an array to hold the FFT results
fft_along_x=[]
for i in range(Ny):
    _U1D=Z[i,:,frame_start]    
    f,e = fftjn(_U1D, _X, 1/(_X[1]-_X[0]), 0.5, detrend=True, window=True)
    fft_along_x.append(e)
fft_along_x=np.array(fft_along_x)

for indx in range(frame_start+1,frame_end):
    _fft_along_x=[]
    for i in range(Ny):
        _U1D=Z[i,:,indx]    
        f,e = fftjn(_U1D, _X, 1/(_X[1]-_X[0]), 0.5, detrend=True, window=True)
        _fft_along_x.append(e)
    fft_along_x+=np.array(_fft_along_x)

fft_along_x/=(frame_end-frame_start)

fgrid,ygrid=np.meshgrid(f,_Y)
plt.figure()
plt.contourf(fgrid,ygrid,fft_along_x, 50, cmap='inferno', zorder =1)
plt.xlabel('Frequency (1/m)')
plt.ylabel('y (m)')
plt.title('FFT along x-axis')

fft_along_x_stacked = np.mean(fft_along_x, axis=0)
plt.figure()
plt.plot(f, fft_along_x_stacked)
plt.xlabel('Frequency (1/m)')
plt.ylabel('Average FFT Magnitude')
plt.title('Average FFT along x-axis')


# np.savez(output_file,
#          dirResults=dirResults, 
#          PIV_file=PIV_file, 
#          dt=dt, 
#          xscale=xscale, 
#          yscale=yscale, 
#          doCalibration=doCalibration, 
#          pts_ROI=pts_ROI, 
#          isROISelection=isROISelection, 
#          isROIextraction=isROIextraction, 
#          frame_start=frame_start, 
#          frame_end=frame_end, 
#          t=t, 
#          fgrid=fgrid,ygrid=ygrid,fft_along_x=fft_along_x)

Text(0.5, 1.0, 'Average FFT along x-axis')

## Vertical FFT

### Single frame

In [10]:
frame_indx=0
output_file=dirResults+'FFT_vertical_singleFrame.npz'

_X=X[0, :]
_Y=Y[:, 0]
Ny=len(_Y)
Nx=len(_X)

Z=V[:,:,frame_indx]#np.sqrt(U**2+V**2)

fft_along_y=[]

for i in range(Nx):
    _U1D=Z[:,i]    
    f,e = fftjn(_U1D, _Y, 1/(_Y[1]-_Y[0]), 0.5, detrend=True, window=True)
    fft_along_y.append(e)

fft_along_y=np.array(fft_along_y)
fgrid,xgrid=np.meshgrid(f,_X)
plt.figure()
plt.contourf(fgrid,xgrid,fft_along_y, 50, cmap='inferno', zorder =1)
plt.xlabel('Frequency (1/m)')
plt.ylabel('x (m)')
plt.title('FFT along y-axis')

fft_along_y_stacked = np.mean(fft_along_y, axis=0)
plt.figure()
plt.plot(f, fft_along_y_stacked)
plt.xlabel('Frequency (1/m)')
plt.ylabel('Average FFT Magnitude')
plt.title('Average FFT along y-axis')

Text(0.5, 1.0, 'Average FFT along y-axis')

### Averaged in time

In [11]:
frame_start=0
frame_end=nframes-1
output_file=dirResults+'FFT_vertical_timeAveraged.npz'

_X=X[0, :]
_Y=Y[:, 0]
Ny=len(_Y)
Nx=len(_X)

Z=V

fft_along_y=[]
for i in range(Nx):
    _U1D=Z[:,i,frame_start]    
    f,e = fftjn(_U1D, _Y, 1/(_Y[1]-_Y[0]), 0.5, detrend=True, window=True)
    fft_along_y.append(e)
fft_along_y=np.array(fft_along_y)


for indx in range(frame_start+1,frame_end):
    _fft_along_y=[]
    for i in range(Nx):
        _U1D=Z[:,i,indx]    
        f,e = fftjn(_U1D, _Y, 1/(_Y[1]-_Y[0]), 0.5, detrend=True, window=True)
        _fft_along_y.append(e)
    fft_along_y+=np.array(_fft_along_y)

fft_along_y/=(frame_end-frame_start)

fgrid,xgrid=np.meshgrid(f,_X)
plt.figure()
plt.contourf(fgrid,xgrid,fft_along_y, 50, cmap='inferno', zorder =1)
plt.xlabel('Frequency (1/m)')
plt.ylabel('x (m)')
plt.title('FFT along y-axis')

fft_along_y_stacked = np.mean(fft_along_y, axis=0)
plt.figure()
plt.plot(f, fft_along_y_stacked)
plt.xlabel('Frequency (1/m)')
plt.ylabel('Average FFT Magnitude')
plt.title('Average FFT along y-axis')

Text(0.5, 1.0, 'Average FFT along y-axis')

## Look at the velocity along x at mid-height of the ROI and along y at mid-horizontal position.

In [57]:
_X=X[Ny//2, :]
_U1Dx=U[Ny//2, :,frame_indx]
_Y=Y[:, Nx//2]
_U1Dy=U[:, Nx//2,frame_indx]

plt.figure()
plt.plot(_X, _U1Dx, label='U along x-axis')
plt.plot(_Y, _U1Dy, label='U along y-axis')
plt.xlabel('x (m)')
plt.ylabel('U (m/s)')
plt.title('U profile')
plt.legend()

# 2D FFT

In [ ]:


_X=X[0, :]
_Y=Y[:, 0]
Ny=len(_Y)
Nx=len(_X)

Z=V[:,:,50]
kx,ekx,ky,eky,s=densitySpectrum2D(_X,_Y,Z)

_kx,_ky=np.meshgrid(np.abs(kx),np.abs(ky))
fig, ax = plt.subplots(figsize=(10, 6))
contour = ax.contourf(_kx, _ky, s, 50, cmap='inferno', zorder=1)
cbar = fig.colorbar(contour, ax=ax)
cbar.set_label('Spectral power')
ax.set_xlabel('kx (1/m)')
ax.set_ylabel('ky (1/m)')
ax.set_title('2D Density Spectrum')
# ax.set_xlim(0, 20)
# ax.set_ylim(0, 20)

Text(0.5, 1.0, '2D Density Spectrum')

# Plot the resonance curve

In [14]:
file = '/Users/jeromenoir/Documents/MyDocuments/LOCAL_PROJECT/TOPOGRAPHY_LIBRATION/CylinderExperimentsGMA/k20_bottomOnly/postprocessing_results_Jerome.xlsx'
df = pd.read_excel(file, skiprows=1, header=0)
data = df.to_numpy()
print('header',data[0,:])
mean_Ekin=data[1:-5,6]
std_Ekin=data[1:-5,7]
flib=data[1:-5,4]
frot=data[1:-5,3]
fstar=flib/frot


plt.figure()
plt.plot(fstar,mean_Ekin)
plt.yscale('log')
plt.xlabel('flib/frot')
plt.ylabel('<Ekin>')

plt.figure()
plt.plot(fstar,std_Ekin)
plt.yscale('log')
plt.xlabel('flib/frot')
plt.ylabel('std Ekin')

header ['R(mm)' 'h' 'k' 'frot (Hz)' 'flib (Hz)' 'dphi (deg)' '<Ekin>(m2/s2)'
 'Ekin_std (m2/s2)']


Text(0, 0.5, 'std Ekin')